<a href="https://www.kaggle.com/code/jakomina/ev-baseline-ipynb?scriptVersionId=353146731" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ⚡ EV Purchase Prediction — Baseline CatBoost + LightGBM 

**Task:** Tabular binary classification to predict the probability of `Will_Buy_EV`.

---

## 📊 Validation Strategy
* **Method:** `StratifiedKFold` (5 folds) to maintain the target distribution ($1s$ and $0s$) across all splits.
* **Metric:** ROC AUC *(Confirm in the competition Evaluation tab; update `CFG.METRIC` if necessary)*.

> **Key Rule:** Out-of-Fold (OOF) CV score is our primary compass. The Public Leaderboard represents only a small slice of the test set and should never drive architectural decisions.

---

## ⚙️ Feature Engineering & Leakage Prevention
* **Row-by-Row Transformations:** All calculated features are applied strictly instance-by-instance without global dataset aggregations, avoiding information leakage across folds.
* **Target Statistics:** Any categorical target encoding is handled safely inside CatBoost’s internal training loop per fold via ordered target statistics.

---

## 🚀 Ensembling & Evaluation
* **Blending Strategy:** Out-of-fold probability weighted average, with optimal weights tuned exclusively on OOF predictions.
* **No Threshold Calibration:** Since ROC AUC evaluates the global ranking order of predicted probabilities rather than hard class assignments, probability threshold optimization is intentionally omitted.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv


In [7]:
import importlib.util, subprocess, sys
for pkg in ["catboost", "lightgbm"]:        
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import gc, os, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def has_cuda() -> bool:
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

In [8]:
# %% [A] Setup, Loading, Diagnostics, and Features
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss


def find_data_dir() -> Path:
    root = Path("/kaggle/input")
    if not root.exists():
        return Path(".")
    # Prioritize the competition folder if multiple train.csv files are mounted
    hits = sorted(root.rglob("train.csv"), key=lambda p: "playground-series-s6e9" not in str(p))
    return hits[0].parent if hits else Path(".")

class CFG:
    DEBUG      = False          # True = subsample of 20k rows and fewer iterations for fast testing
    RUN_CHECKS = True           # Data diagnostics (ID, duplicates, drift); disable after the first run
    RUN_CB     = False          # False = LightGBM only during feature experimentation (~20 min faster)
    SEED       = 42
    N_FOLDS    = 5
    METRIC     = "auc"          # "auc" or "logloss" (based on the competition Evaluation tab)
    DATA_DIR   = find_data_dir()
    OUT_DIR    = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    TARGET     = "Will_Buy_EV"
    ID         = "id"
    USE_GPU    = has_cuda()     # Assuming has_cuda() is defined in the environment
    EARLY_STOP = 200
    CB_ITERS   = 3000
    LGB_ITERS  = 5000
    CAT_COLS   = [
        "Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
        "Subsidy_Available", "Range_Anxiety_Level", "Environmental_Concern_Level",
    ]

if CFG.DEBUG:
    CFG.CB_ITERS, CFG.LGB_ITERS, CFG.EARLY_STOP = 300, 300, 50

seed_everything(CFG.SEED)

def score(y, p):
    return roc_auc_score(y, p) if CFG.METRIC == "auc" else log_loss(y, p)

def reduce_mem(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].astype("float32")
    for c in df.select_dtypes("int64").columns:
        if c != CFG.ID:
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

# --- Loading ---
train = reduce_mem(pd.read_csv(CFG.DATA_DIR / "train.csv"))
test  = reduce_mem(pd.read_csv(CFG.DATA_DIR / "test.csv"))
sub   = pd.read_csv(CFG.DATA_DIR / "sample_submission.csv")

assert CFG.TARGET in train.columns, f"Target column '{CFG.TARGET}' not found in train data."
assert set(test.columns) == set(train.columns) - {CFG.TARGET}, "Train and test datasets do not have matching columns."

if not pd.api.types.is_numeric_dtype(train[CFG.TARGET]):
    train[CFG.TARGET] = train[CFG.TARGET].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
assert train[CFG.TARGET].isin([0, 1]).all(), "Target contains values other than Yes/No."
train[CFG.TARGET] = train[CFG.TARGET].astype("int8")

if CFG.DEBUG:
    train = train.sample(n=min(20_000, len(train)), random_state=CFG.SEED).reset_index(drop=True)

print("train:", train.shape, "| test:", test.shape, "| sample_submission:", sub.shape)
print("Missing values (train):", int(train.isna().sum().sum()), "| (test):", int(test.isna().sum().sum()))
print(f"Positive rate: {train[CFG.TARGET].mean():.4f} | Constant in sample_submission: {sub.iloc[0, 1]:.4f}")

# --- Diagnostics (on raw columns, prior to feature engineering) ---
if CFG.RUN_CHECKS:
    raw = [c for c in test.columns if c != CFG.ID]
    auc_id = roc_auc_score(train[CFG.TARGET], train[CFG.ID])
    print(f"\nID-only AUC: {auc_id:.4f} | |AUC-0.5| = {abs(auc_id - 0.5):.4f}  (should be ~0)")
    
    dups = train[raw].merge(test[raw].drop_duplicates(), how="inner").shape[0]
    print(f"Train rows identical to a test row: {dups:,}")
    
    for c in [c for c in CFG.CAT_COLS if c in test.columns]:
        unseen = set(test[c].unique()) - set(train[c].unique())
        if unseen:
            print(f"  ⚠ {c}: Categories exclusive to test set → {unseen}")
            
    num_raw = train[raw].select_dtypes("number").columns
    drift = ((train[num_raw].mean() - test[num_raw].mean()) / train[num_raw].std().replace(0, np.nan)).abs()
    print("Train→Test drift in σ (> 0.05 is highly suspicious):")
    print(drift.sort_values(ascending=False).round(3).to_string())

# --- Feature Engineering ---
ANXIETY_ORDER = {"low": 0, "medium": 1, "high": 2}

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    cols = set(out.columns)

    if {"Charging_Stations_Near_Home", "Charging_Stations_Near_Work"} <= cols:
        out["Total_Stations"] = out["Charging_Stations_Near_Home"] + out["Charging_Stations_Near_Work"]

    if {"Annual_Income_USD", "Number_of_Cars_Owned"} <= cols:
        out["Income_per_Car"] = out["Annual_Income_USD"] / (out["Number_of_Cars_Owned"] + 1)

    if "Range_Anxiety_Level" in cols:
        ral = out["Range_Anxiety_Level"]
        # Use as-is if already numeric; otherwise, map string values to Low/Medium/High integers
        anx = ral if pd.api.types.is_numeric_dtype(ral) else ral.astype(str).str.strip().str.lower().map(ANXIETY_ORDER)
        out["Anxiety_Ord"] = anx.astype("float32")
        if "Daily_Commute_km" in cols:
            out["Commute_x_Anxiety"] = out["Daily_Commute_km"] * (out["Anxiety_Ord"] + 1)
            
    if "Environmental_Concern_Level" in cols and pd.api.types.is_numeric_dtype(out["Environmental_Concern_Level"]):
        out["Env_Concern_Num"] = out["Environmental_Concern_Level"].astype("float32")   
        
    return out

train = make_features(train)
test  = make_features(test)

if "Anxiety_Ord" in train and train["Anxiety_Ord"].isna().all():
    print("⚠ Anxiety_Ord is completely NaN: verify the actual values of Range_Anxiety_Level.")

missing_cats = [c for c in CFG.CAT_COLS if c not in train.columns]
if missing_cats:
    print("⚠ Categorical columns not found (skipping):", missing_cats)
CFG.CAT_COLS = [c for c in CFG.CAT_COLS if c in train.columns]

FEATURES  = [c for c in test.columns if c != CFG.ID]
NUM_FEATS = [c for c in FEATURES if c not in CFG.CAT_COLS]

# Common categorical vocabulary (train ∪ test): features names only, target excluded → prevents data leakage
for c in CFG.CAT_COLS:
    cats = sorted(set(train[c].astype(str)) | set(test[c].astype(str)))
    train[c] = pd.Categorical(train[c].astype(str), categories=cats)
    test[c]  = pd.Categorical(test[c].astype(str),  categories=cats)

print(f"\n{len(FEATURES)} features ({len(CFG.CAT_COLS)} categorical, {len(NUM_FEATS)} numeric):", FEATURES)
print(f"Data: {CFG.DATA_DIR.resolve()} | GPU: {CFG.USE_GPU} | DEBUG: {CFG.DEBUG} | CatBoost: {CFG.RUN_CB} | Metric: {CFG.METRIC}")

train: (668665, 15) | test: (286571, 14) | sample_submission: (286571, 2)
Missing values (train): 0 | (test): 0
Positive rate: 0.1746 | Constant in sample_submission: 0.1746

ID-only AUC: 0.5000 | |AUC-0.5| = 0.0000  (should be ~0)
Train rows identical to a test row: 0
Train→Test drift in σ (> 0.05 is highly suspicious):
Number_of_Cars_Owned           0.005
Charging_Stations_Near_Work    0.004
Charging_Stations_Near_Home    0.002
Age                            0.002
Annual_Income_USD              0.001
Environmental_Concern_Level    0.001
Daily_Commute_km               0.000

18 features (7 categorical, 11 numeric): ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Total_Stations', 'Income_per_Car', 'Anxiety_Ord', 'Commute_x_Anxiety', 'Env_Concern_Num']
Data: 

In [4]:
# %% [B] Cross-Validation Models
# Fixed folds, calculated only once: all models will use exactly the same partitions
FOLD = np.full(len(train), -1, dtype="int8")
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
for k, (_, va) in enumerate(skf.split(train, train[CFG.TARGET])):
    FOLD[va] = k


def run_cv(fit_fn, name):
    X, y, X_te = train[FEATURES], train[CFG.TARGET].values, test[FEATURES]
    oof, pred = np.zeros(len(train)), np.zeros(len(test))
    t0 = time.time()
    for k in range(CFG.N_FOLDS):
        tr, va = np.where(FOLD != k)[0], np.where(FOLD == k)[0]
        p_va, p_te = fit_fn(X.iloc[tr], y[tr], X.iloc[va], y[va], X_te)
        oof[va] = p_va
        pred += p_te / CFG.N_FOLDS
        print(f"[{name}] fold {k + 1}: {CFG.METRIC} = {score(y[va], p_va):.5f}")
        gc.collect()
    print(f"[{name}] OOF {CFG.METRIC} = {score(y, oof):.5f} | {(time.time() - t0) / 60:.1f} min\n")
    return oof, pred


def fit_lgbm(X_tr, y_tr, X_va, y_va, X_te):
    model = lgb.LGBMClassifier(
        n_estimators=CFG.LGB_ITERS, learning_rate=0.03, num_leaves=63, min_child_samples=50,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=CFG.SEED, verbose=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              eval_metric="auc" if CFG.METRIC == "auc" else "binary_logloss",
              callbacks=[lgb.early_stopping(CFG.EARLY_STOP, verbose=False)])
    return (model.predict_proba(X_va, num_iteration=model.best_iteration_)[:, 1],
            model.predict_proba(X_te, num_iteration=model.best_iteration_)[:, 1])


def to_code(df_in):
    # Encode categories as integers (not strings): reduces RAM usage and speeds up training
    return df_in.assign(**{c: df_in[c].cat.codes.astype("int32") for c in CFG.CAT_COLS})


def fit_catboost(X_tr, y_tr, X_va, y_va, X_te):
    params = dict(
        iterations=CFG.CB_ITERS, learning_rate=0.08, depth=6, l2_leaf_reg=3,
        max_ctr_complexity=1,   # Disable category combinations: ~1.8x faster
        border_count=128,
        loss_function="Logloss",
        eval_metric="AUC" if CFG.METRIC == "auc" else "Logloss",
        od_type="Iter", od_wait=CFG.EARLY_STOP, use_best_model=True,
        random_seed=CFG.SEED, verbose=250,
    )
    if CFG.USE_GPU:
        params.update(task_type="GPU", devices="0")
    else:
        params.update(thread_count=-1)

    model = CatBoostClassifier(**params)
    X_va_cb = to_code(X_va)
    model.fit(Pool(to_code(X_tr), y_tr, cat_features=CFG.CAT_COLS),
              eval_set=Pool(X_va_cb, y_va, cat_features=CFG.CAT_COLS))
    fit_catboost.last_model = model
    return model.predict_proba(X_va_cb)[:, 1], model.predict_proba(TEST_CB)[:, 1]


OOF, PRED = {}, {}
OOF["LightGBM"], PRED["LightGBM"] = run_cv(fit_lgbm, "LightGBM")

if CFG.RUN_CB:
    TEST_CB = to_code(test[FEATURES])  # Converted only once prior to CV loop
    OOF["CatBoost"], PRED["CatBoost"] = run_cv(fit_catboost, "CatBoost")

[LightGBM] fold 2: auc = 0.94157
[LightGBM] fold 3: auc = 0.94289
[LightGBM] fold 4: auc = 0.94235
[LightGBM] fold 5: auc = 0.94186
[LightGBM] OOF auc = 0.94184 | 5.7 min



In [5]:
# %% [C] Blending, Submission, and OOF Saving
y = train[CFG.TARGET].values
results = pd.Series({name: score(y, p) for name, p in OOF.items()}, dtype=float)

if "CatBoost" in OOF:
    # Search for the optimal blending weight between 0 and 1
    ws = np.linspace(0, 1, 21)
    sc = np.array([score(y, w * OOF["LightGBM"] + (1 - w) * OOF["CatBoost"]) for w in ws])
    i = sc.argmax() if CFG.METRIC == "auc" else sc.argmin()
    w = ws[i]
    pred_blend = w * PRED["LightGBM"] + (1 - w) * PRED["CatBoost"]
    results[f"Blend (w_cb={1 - w:.2f})"] = sc[i]
else:
    pred_blend = PRED["LightGBM"]

print(f"CV {CFG.METRIC}:")
print(results.round(5).to_string())

# --- Submission ---
col = sub.columns[1]
submission = sub[[CFG.ID]].merge(
    pd.DataFrame({CFG.ID: test[CFG.ID].values, col: pred_blend}), on=CFG.ID, how="left"
)

# Sanity checks prior to saving
assert submission.shape == sub.shape, "Rows/columns mismatch with sample_submission."
assert list(submission.columns) == list(sub.columns), "Columns mismatch with sample_submission."
assert submission[col].notna().all(), "Null predictions found."
assert submission[col].between(0, 1).all(), "Probabilities found outside the [0, 1] range."

submission.to_csv(CFG.OUT_DIR / "submission.csv", index=False)

# --- OOF and test predictions: useful for future ensembling without retraining ---
pd.DataFrame({
    CFG.ID: train[CFG.ID].values, 
    "fold": FOLD,
    **{f"oof_{k.lower()}": v for k, v in OOF.items()}
}).to_csv(CFG.OUT_DIR / "oof_baseline.csv", index=False)

pd.DataFrame({
    CFG.ID: test[CFG.ID].values,
    **{f"pred_{k.lower()}": v for k, v in PRED.items()}
}).to_csv(CFG.OUT_DIR / "pred_baseline.csv", index=False)

print(f"\n✅ submission.csv saved in {CFG.OUT_DIR.resolve()} | rows: {len(submission):,}")
print(f"Prediction mean: {submission[col].mean():.4f} vs Positive rate in train: {y.mean():.4f}")
submission.head()

CV auc:
LightGBM    0.94184

✅ submission.csv saved in /kaggle/working | rows: 286,571
Prediction mean: 0.1746 vs Positive rate in train: 0.1746


,id,Will_Buy_EV
0,668665,0.009823
1,668666,0.018761
2,668667,0.005818
3,668668,0.003057
4,668669,0.016297


In [10]:
# %% [D] Dark Theme Visualizations (Run after A, B, and C)
import math
import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# --- Dark theme: surface, inks, and 2 colorblind-friendly series colors ---
DARK = dict(surface="#1a1a19", ink="#ffffff", ink2="#c3c2b7", muted="#898781",
            grid="#2c2c2a", axis="#383835", s1="#3987e5", s2="#d95926")
NO, YES = "Won't Buy (0)", "Will Buy (1)"

_axis = dict(gridcolor=DARK["grid"], gridwidth=1, linecolor=DARK["axis"], zeroline=False, automargin=True,
             ticks="", tickfont=dict(color=DARK["muted"], size=11), title=dict(font=dict(color=DARK["muted"])))

pio.templates["ev_dark"] = go.layout.Template(layout=dict(
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", size=13, color=DARK["ink2"]),
    title=dict(font=dict(size=18, color=DARK["ink"]), x=0.01, xanchor="left",
               y=1, yanchor="top", yref="container", pad=dict(t=20)),
    paper_bgcolor=DARK["surface"], plot_bgcolor=DARK["surface"],
    colorway=[DARK["s1"], DARK["s2"]],
    xaxis=_axis, yaxis=_axis,
    legend=dict(orientation="h", x=0.01, xanchor="left", y=1.0, yanchor="bottom",
                bgcolor="rgba(0,0,0,0)", font=dict(color=DARK["ink2"])),
    hoverlabel=dict(bgcolor="#262624", bordercolor=DARK["axis"], font=dict(color=DARK["ink"])),
    annotationdefaults=dict(font=dict(color=DARK["ink2"], size=13)),
    margin=dict(l=70, r=30, t=110, b=50),
))

# Apply rounded bar corners if Plotly version supports it
if tuple(int(v) for v in plotly.__version__.split(".")[:2]) >= (5, 19):
    pio.templates["ev_dark"].layout.barcornerradius = 4

pio.templates.default = "ev_dark"
CONFIG = {"displaylogo": False}

def plot_title(main_text, sub_text):
    return f"{main_text}<br><span style='font-size:13px;color:{DARK['ink2']}'>{sub_text}</span>"

def natural_sort(idx):
    s = pd.Series(idx.astype(str), index=idx)
    try:
        return s.astype(float).sort_values().index                     # e.g., 1, 2, ..., 10
    except ValueError:
        low = s.str.strip().str.lower()
        if low.isin(ANXIETY_ORDER.keys()).all():
            return low.map(ANXIETY_ORDER).sort_values().index          # Low, Medium, High
        return s.sort_values().index


y_true = train[CFG.TARGET].values
base = y_true.mean()
RAW = [c for c in pd.read_csv(CFG.DATA_DIR / "test.csv", nrows=0).columns if c != CFG.ID]
RAW_NUM = [c for c in RAW if c not in CFG.CAT_COLS]


# ---------- 1) Purchase Rate by Category ----------
ncol, nrow = 2, math.ceil(len(CFG.CAT_COLS) / 2)
n_cat = [train[c].nunique() for c in CFG.CAT_COLS]
row_w = [max(3, *n_cat[r * ncol:(r + 1) * ncol]) for r in range(nrow)]   # Adjust height for many categories
H1 = 34 * sum(row_w) + 70 * nrow + 150
fig = make_subplots(rows=nrow, cols=ncol, subplot_titles=CFG.CAT_COLS, row_heights=row_w,
                    horizontal_spacing=0.2, vertical_spacing=60 / H1)
xmax = 0
for i, c in enumerate(CFG.CAT_COLS):
    r, k = i // ncol + 1, i % ncol + 1
    g = train.groupby(c, observed=True)[CFG.TARGET].agg(["mean", "size"])
    g = g.loc[natural_sort(g.index)]
    xmax = max(xmax, g["mean"].max())
    fig.add_bar(y=g.index.astype(str), x=g["mean"], orientation="h", marker_color=DARK["s1"],
                width=min(0.8, 22 / (34 * row_w[r - 1] / len(g))), customdata=g["size"], showlegend=False,
                hovertemplate="<b>%{y}</b><br>Purchase Rate: %{x:.1%}<br>n = %{customdata:,}<extra></extra>",
                row=r, col=k)
    fig.add_vline(x=base, line=dict(color=DARK["muted"], width=1), row=r, col=k)

fig.update_xaxes(tickformat=".0%", range=[0, xmax * 1.1])
fig.update_yaxes(showgrid=False, type="category", autorange="reversed")
fig.update_layout(height=H1, bargap=0,
                  title=plot_title("Purchase Rate by Category",
                                   f"The gray line is the global rate ({base:.1%}). Hover to see sample size (n)."))
fig.show(config=CONFIG)


# ---------- 2) Numeric Variables: Distribution by Class ----------
nrow = math.ceil(len(RAW_NUM) / 2)
fig = make_subplots(rows=nrow, cols=2, subplot_titles=RAW_NUM,
                    horizontal_spacing=0.1, vertical_spacing=0.16 / max(nrow - 1, 1) * 2)
for i, c in enumerate(RAW_NUM):
    r, k = i // 2 + 1, i % 2 + 1
    col = train[c]
    nu = col.nunique()
    is_discrete = nu <= 100 and np.allclose(col.dropna().values[:10_000] % 1, 0)  # Integers: one point per value
    
    if is_discrete:
        vals = np.sort(col.dropna().unique())
        ys = [col[y_true == cls].value_counts(normalize=True).reindex(vals, fill_value=0).values for cls in (0, 1)]
        xs = vals
    else:
        edges = np.linspace(*np.percentile(col.dropna(), [0.5, 99.5]), 61)
        xs = (edges[:-1] + edges[1:]) / 2
        ys = [np.histogram(col[y_true == cls].dropna(), edges)[0] / (y_true == cls).sum() for cls in (0, 1)]
        
    for cls, name, color in [(0, NO, DARK["s1"]), (1, YES, DARK["s2"])]:
        fig.add_scatter(x=xs, y=ys[cls], name=name, legendgroup=name, showlegend=(i == 0),
                        mode="lines+markers" if nu <= 25 else "lines",
                        line=dict(color=color, width=2),
                        marker=dict(size=8, color=color, line=dict(width=2, color=DARK["surface"])),
                        hovertemplate=f"{name}: %{{y:.1%}}<extra></extra>", row=r, col=k)
    if is_discrete and nu <= 12:
        fig.update_xaxes(dtick=1, row=r, col=k)

fig.update_yaxes(tickformat=".0%", rangemode="tozero")
fig.update_xaxes(showspikes=True, spikemode="across", spikethickness=1, spikecolor=DARK["muted"], spikedash="solid")
H2 = 260 * nrow + 150
fig.update_layout(height=H2, hovermode="x unified",
                  legend=dict(x=1, xanchor="right", y=1 + 45 / (H2 - 160), yanchor="bottom"),
                  title=plot_title("Numeric Variables by Class",
                                   "Percentage of each class per value/interval. Where curves separate, the variable discriminates well."))
fig.show(config=CONFIG)


# ---------- 3) OOF Prediction Probability by Class ----------
if "CatBoost" in OOF:
    oof_plot, plot_name = w * OOF["LightGBM"] + (1 - w) * OOF["CatBoost"], f"Blend (w_cb={1 - w:.2f})"
else:
    oof_plot, plot_name = OOF["LightGBM"], "LightGBM"

edges = np.linspace(0, 1, 51)
xs = (edges[:-1] + edges[1:]) / 2
fig = go.Figure()

for cls, name, color, fill in [(0, NO, DARK["s1"], "rgba(57,135,229,0.10)"),
                               (1, YES, DARK["s2"], "rgba(217,89,38,0.10)")]:
    h = np.histogram(oof_plot[y_true == cls], edges)[0] / (y_true == cls).sum()
    fig.add_scatter(x=xs, y=h, name=name, mode="lines", line=dict(color=color, width=2),
                    fill="tozeroy", fillcolor=fill, hovertemplate=f"{name}: %{{y:.1%}}<extra></extra>")

fig.update_xaxes(title_text="Predicted Probability (OOF)", tickformat=".0%", range=[0, 1],
                 showspikes=True, spikemode="across", spikethickness=1, spikecolor=DARK["muted"], spikedash="solid")
fig.update_yaxes(title_text="% of Class", tickformat=".0%", rangemode="tozero")
fig.update_layout(height=440, hovermode="x unified",
                  title=plot_title(f"Class Separation · {plot_name}",
                                   f"OOF {CFG.METRIC} = {score(y_true, oof_plot):.5f}. Less overlap between curves = better AUC."))
fig.show(config=CONFIG)


# ---------- 4) Feature Importance (LightGBM, Gain) ----------
m = getattr(fit_lgbm, "last_model", None)
if m is None:   # Fast model specifically for feature importance (~30 s)
    s = train.sample(n=min(200_000, len(train)), random_state=CFG.SEED)
    import lightgbm as lgb
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=63, min_child_samples=50,
                           colsample_bytree=0.8, random_state=CFG.SEED, verbose=-1)
    m.fit(s[FEATURES], s[CFG.TARGET])
    source = f"Fast model with {len(s):,} rows"
else:
    source = "Last CV fold"

imp = pd.Series(m.booster_.feature_importance("gain"), index=FEATURES)
imp = (imp / imp.sum()).sort_values()

fig = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation="h", marker_color=DARK["s1"], width=0.6,
                       hovertemplate="<b>%{y}</b><br>%{x:.1%} of total gain<extra></extra>"))
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(showgrid=False)
fig.update_layout(height=max(420, 30 * len(imp) + 160),
                  title=plot_title("Feature Importance · LightGBM (Gain)",
                                   f"Proportion of total gain ({source}). Bottom features contribute very little."))
fig.show(config=CONFIG)